# 2장 모델 품질 증거 비교

## Goal

모델을 새로 튜닝하지 않고 준비된 canonical evidence에서 baseline, Candidate A, Candidate B의 metric과 release decision을 비교합니다. Candidate A를 보류하고 Candidate B를 승인한 근거를 수치로 설명하는 것이 목표입니다.

## Setup

입력은 `reference/evidence/model/revisions/v2/canonical-benchmark.json`입니다. 이 파일은 sealed test를 한 번 평가한 결과이며 이 Notebook은 evidence를 읽기만 합니다.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "tta-aiqa repository root를 찾지 못했습니다."
ROOT

PosixPath('/Users/seungbaeji/Workspace/tta/tta-aiqa')

In [2]:
import json
import pandas as pd

EVIDENCE_PATH = ROOT / "reference/evidence/model/revisions/v2/canonical-benchmark.json"
evidence = json.loads(EVIDENCE_PATH.read_text(encoding="utf-8"))
assert evidence["sealed_test"]["status"] == "evaluated_once"
assert evidence["deployment_allowed"] is True
EVIDENCE_PATH

PosixPath('/Users/seungbaeji/Workspace/tta/tta-aiqa/reference/evidence/model/revisions/v2/canonical-benchmark.json')

## Steps

### 1. 공통 test metric 비교

In [3]:
decisions = {item["profile"]: item["decision"] for item in evidence["decisions"]}
rows = []
for profile in evidence["profiles"]:
    metrics = profile["metrics"]
    rows.append({
        "profile": profile["profile"],
        "threshold": profile["threshold"],
        "pr_auc": metrics["pr_auc"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "false_negative": metrics["false_negative"],
        "decision": decisions.get(profile["profile"], "REFERENCE"),
    })
comparison = pd.DataFrame(rows).set_index("profile")
comparison.round(4)

,threshold,pr_auc,precision,recall,false_negative,decision
profile,,,,,,
baseline,0.50,0.5244,0.5652,0.2364,42,REFERENCE
candidate-a,0.40,0.5942,0.7727,0.3091,38,HOLD
candidate-b,0.35,0.5743,0.3793,0.8000,11,APPROVE


### 2. Release guardrail 확인

PR-AUC 하나만으로 승인하지 않습니다. Recall, uncertainty lower bound, Precision floor와 FN 감소 조건을 함께 확인합니다.

In [4]:
checks = pd.DataFrame({
    item["profile"]: item["checks"] for item in evidence["decisions"]
}).T
checks

,false_negative_reduction,pr_auc_vs_baseline,precision_floor,recall_guardrail,recall_uncertainty
candidate-a,False,True,True,False,False
candidate-b,True,True,True,True,True


## Checks

In [5]:
assert decisions == {"candidate-a": "HOLD", "candidate-b": "APPROVE"}
assert comparison.loc["candidate-b", "recall"] > comparison.loc["baseline", "recall"]
assert comparison.loc["candidate-b", "false_negative"] < comparison.loc["baseline", "false_negative"]
assert not checks.loc["candidate-a"].all()
assert checks.loc["candidate-b"].all()
print("Canonical model evidence checks passed.")

Canonical model evidence checks passed.


## Next Steps

MLflow UI에서 같은 profile의 parameter, validation metric, dataset lineage를 확인합니다. 이 Notebook에서 feature나 threshold를 변경하지 않습니다.